# PRÁCTICA BACKTESTING AVANZADO

##### Realizado por Mateo Santos

In [1]:
# Importamos las librerías
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import pyarrow
import pyarrow.parquet as pq
import time

# Fijamos la semilla
np.random.seed(42)

In [2]:
# Definición de variables globales
capital = 250000
#START_DATE = '2015-01-01'
START_DATE = '2014-01-01'
END_DATE = '2026-01-31'
NUM_ACTIVOS = 20
capital_por_activo = capital/NUM_ACTIVOS

## Notebook 1: Carga de Datos

Leemos los datos de las cotizaciones de precios del universo de activos. Podemos apreciar que **aparecen en formato largo**. Es decir, **las claves primarias de cada fila son la fecha y el activo (symbol)**. Habrá que modificar la estructura de los datos y hacer una serie de filtrados para tener el dataset final con el que trabajar.

Se realiza un procesamiento de los datos al eliminar los tickers duplicados. Nos quedamos con los tickers que tienen un mayor volumen, que implica que son activos más líquidos y, por tanto, más fáciles de negociar.

In [3]:
# 1. Carga de datos
precios_activos = pq.read_table('sp500_history.parquet').to_pandas()

# 2. Calcular el volumen total por Ticker para cada Empresa. Nos dice qué ticker es el "dominante" históricamente
volumen_por_ticker = precios_activos.groupby(['security_name', 'symbol'])['volume'].sum().reset_index()

# 3. Para cada empresa, encontrar el registro con el volumen máximo
# Ordenamos por volumen descendente y nos quedamos con el primero por cada nombre de empresa
tickers_ganadores = volumen_por_ticker.sort_values('volume', ascending=False) \
                                      .drop_duplicates(subset=['security_name'], keep='first')

# 4. Filtrar el DataFrame original usando solo los pares (Empresa, Ticker) con mayor volumen
# Hacemos un 'merge' tipo inner para descartar los tickers secundarios
precios_activos_consistentes = pd.merge(precios_activos, 
    tickers_ganadores[['security_name', 'symbol']], on=['security_name', 'symbol'], how='inner')

# 5. Finalmente, eliminamos duplicados por fecha por si acaso (limpieza estándar)
precios_activos = precios_activos_consistentes.drop_duplicates(subset=['security_name', 'date'])

precios_activos.head()

,date,symbol,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close
0,1999-11-18,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,27.188307,29.877260,23.901808,25.545057,74862288.0,42.7500
1,1999-11-19,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.657097,25.694445,23.789768,24.349968,18236110.0,40.7500
2,1999-11-22,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,24.686087,26.067909,23.939156,26.067909,7874048.5,43.6250
3,1999-11-23,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.395672,26.067909,24.051195,24.051195,7153099.0,40.2500
4,1999-11-24,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,23.976501,25.059551,23.901808,24.536699,5797720.5,41.0625


Leemos los datos de las **cotizaciones diarias del S&P 500** a través de lal librería yfinance. Nos quedamos con los precios de apertura y cierre.

In [4]:
# Leer el archivo de precios del sp500. Nos quedamos con los precios de cierre y de apertura
precios_sp500 = yf.download('^GSPC', start=START_DATE, end=END_DATE)[['Close', 'Open']]
precios_sp500.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,Open
Ticker,^GSPC,^GSPC
Date,,
2014-01-02,1831.979980,1845.859985
2014-01-03,1831.369995,1833.209961
2014-01-06,1826.770020,1832.310059
2014-01-07,1837.880005,1828.709961
2014-01-08,1837.489990,1837.900024
